# Gold Layer — Aggregated NJ Transit Analytics

This notebook builds the gold-layer analytical tables from the silver schema: joins transit trips with weather via the station-region bridge, computes daily/hourly performance and reliability KPIs by line and station, and writes curated aggregate tables to the `workspace.nj_transit_gold` schema.

In [0]:
# --- PySpark (core lakehouse operations) ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException

# --- Python standard libs for date/time and config ---
from datetime import datetime, date
import json

# --- Schema config ---
GOLD_SCHEMA = "workspace.nj_transit_gold"
SILVER_SCHEMA = "workspace.nj_transit_silver"

In [0]:
%sql
SELECT * 
FROM nj_transit_silver.weather
LIMIT 5;

-- SELECT *
-- FROM nj_transit_silver.station_weather_region_bridge
-- LIMIT 5;

-- %sql
-- SELECT *
-- FROM nj_transit_silver.transit_trips
-- LIMIT 5;


In [0]:
%sql

-- Merges transit_trips with station_weather_region_bridge on from_id = STATION_ID, then weather on region_id + hourly actual_time
CREATE OR REPLACE VIEW workspace.nj_transit_gold.enriched_trips AS
SELECT
    t.date,
    t.train_id,
    t.stop_sequence,
    t.`from`,
    t.from_id,
    t.`to`,
    t.to_id,
    t.scheduled_time,
    t.actual_time,
    t.delay_minutes,
    t.status,
    t.line,
    t.type,
    t.is_cancelled,
    b.region_name AS weather_region,
    w.temperature_2m_c,
    w.precipitation_mm,
    w.wind_speed_kmh
FROM workspace.nj_transit_silver.transit_trips t
LEFT JOIN workspace.nj_transit_silver.station_weather_region_bridge b
    ON t.from_id = b.STATION_ID
LEFT JOIN workspace.nj_transit_silver.weather w
    ON b.region_id = w.region_id
    AND date_trunc('hour', t.actual_time) = w.timestamp;

In [0]:
%sql
-- Create a view getting only the needed columns from the silver layer
CREATE OR REPLACE VIEW workspace.nj_transit_gold.station_weather_region AS
SELECT
    STATION_ID,
    STATION_NAME,
    COUNTY,
    LATITUDE,
    LONGITUDE,
    LINE_CODE,
    MUNICIPALITY,
    ADA,
    region_name,
    region_id
FROM workspace.nj_transit_silver.station_weather_region_bridge;

In [0]:
%sql
-- Create a view passing on all of the columns from the silver layer
CREATE OR REPLACE VIEW workspace.nj_transit_gold.line_coordinates AS
SELECT *
FROM workspace.nj_transit_silver.line_coordinates;